## 5 - Data Types & Conversion

### Pandas dtypes
| dtype | Pandas type | Example values |
|-------|------------|----------------|
| `int64` | Integer | 1, 2, 100 |
| `float64` | Float | 3.14, np.nan |
| `object` | String (Python str) | 'Alice', 'Delhi' |
| `bool` | Boolean | True, False |
| `datetime64[ns]` | Timestamp | 2024-01-15 |
| `category` | Categorical | 'Male'/'Female' (stored as ints) |

### Why dtypes matter
- Wrong dtype → operations fail silently or raise errors
- `object` strings can't do `mean()` or comparisons
- `category` dtype uses **much less memory** for low-cardinality columns
- `int64` → `int8` for small integers saves 8× memory

### Conversion methods
| Method | Use case |
|--------|----------|
| `df['col'].astype(int)` | Cast to a specific type |
| `pd.to_numeric(s, errors='coerce')` | Convert to number, NaN on failure |
| `pd.to_datetime(s)` | Convert to datetime |
| `pd.Categorical(s)` | Convert to category |


In [1]:
import pandas as pd
df = pd.read_csv('students.csv')

In [2]:
# Current dtypes 
print('Current dtypes:')
print(df.dtypes)

Current dtypes:
student_id      object
name            object
gender          object
age              int64
study_hours    float64
maths          float64
science        float64
english        float64
compSci        float64
attendance     float64
dtype: object


In [3]:
# astype
df2 = df.copy()
df2['age'] = df2['age'].astype('int8')          # int64 → int8 (saves memory)
df2['maths'] = df2['maths'].astype('float32')   # float64 → float32
print('After downcasting:')
print(df2[['age','maths']].dtypes)

After downcasting:
age         int8
maths    float32
dtype: object


In [5]:
#  Category dtype 
df3 = df.copy()
before_mem = df3['gender'].memory_usage(deep=True)
df3['gender'] = df3['gender'].astype('category')
after_mem  = df3['gender'].memory_usage(deep=True)

print(f'gender column memory before: {before_mem} bytes')
print(f'gender column memory after : {after_mem} bytes')
print(f'Memory saved: {before_mem - after_mem} bytes ({(1-after_mem/before_mem)*100:.0f}%)')
print('Categories:', df3['gender'].cat.categories.tolist())

gender column memory before: 2828 bytes
gender column memory after : 398 bytes
Memory saved: 2430 bytes (86%)
Categories: ['Female', 'Male']


In [9]:
# to_numeric with error handling
messy = pd.Series(['85', '92.5', 'N/A', '78', 'missing', '88'])
converted = pd.to_numeric(messy, errors='coerce')  # bad → NaN
print('Messy series  :', messy.tolist())
print('to_numeric    :', converted.tolist())
print('Mean (clean)  :', converted.mean())

Messy series  : ['85', '92.5', 'N/A', '78', 'missing', '88']
to_numeric    : [85.0, 92.5, nan, 78.0, nan, 88.0]
Mean (clean)  : 85.875


In [11]:
# Overall memory usage comparison
print('Memory before optimisation:', df.memory_usage(deep=True).sum(), 'bytes')

df_opt = df.copy()
for col in ['gender']:            
    df_opt[col] = df_opt[col].astype('category')
for col in ['age']:                        
    df_opt[col] = df_opt[col].astype('int8')
for col in ['maths','science','english','compSci','study_hours','attendance']:
    df_opt[col] = df_opt[col].astype('float32')

print('Memory after  optimisation:', df_opt.memory_usage(deep=True).sum(), 'bytes')
ratio = df.memory_usage(deep=True).sum() / df_opt.memory_usage(deep=True).sum()
print(f'Memory reduction: {ratio:.1f}x smaller')

Memory before optimisation: 10980 bytes
Memory after  optimisation: 7000 bytes
Memory reduction: 1.6x smaller
